# ASVspoof 2019 Logical Access

Official speaker-disjoint partitions; the evaluation partition contains attacks A07 to A19, none of which appear in training (Section 4.4, Tables 9 and 10).

In [ ]:
from pathlib import Path
import pandas as pd

for cand in [Path('/content/LA'), Path('/content/drive/MyDrive/deepfake_finetuning/asvspoof/LA')]:
    print(cand, '->', cand.exists())
LA = Path('/content/LA') if Path('/content/LA').exists() else \
     Path('/content/drive/MyDrive/deepfake_finetuning/asvspoof/LA')
print('using', LA)

for pat in ['*train.trn.txt','*dev.trl.txt','*eval.trl.txt']:
    f = next(LA.rglob(pat), None)
    if f is None: print('MISSING', pat); continue
    df = pd.DataFrame([l.split() for l in open(f)],
                      columns=['speaker','utt_id','_','attack','label'])
    print(f'\n{f.name}: {len(df)} utterances, {df.speaker.nunique()} speakers')
    print(' ', df.label.value_counts().to_dict())
    print('  attacks:', sorted(df.attack.unique()))

for d in sorted(LA.rglob('ASVspoof2019_LA_*')):
    if d.is_dir():
        n = sum(1 for _ in d.rglob('*.flac'))
        if n: print(d.name, n, 'flac')

/content/LA -> False
/content/drive/MyDrive/deepfake_finetuning/asvspoof/LA -> True
using /content/drive/MyDrive/deepfake_finetuning/asvspoof/LA

ASVspoof2019.LA.cm.train.trn.txt: 25380 utterances, 20 speakers
  {'spoof': 22800, 'bonafide': 2580}
  attacks: ['-', 'A01', 'A02', 'A03', 'A04', 'A05', 'A06']

ASVspoof2019.LA.cm.dev.trl.txt: 24844 utterances, 20 speakers
  {'spoof': 22296, 'bonafide': 2548}
  attacks: ['-', 'A01', 'A02', 'A03', 'A04', 'A05', 'A06']

ASVspoof2019.LA.cm.eval.trl.txt: 71237 utterances, 67 speakers
  {'spoof': 63882, 'bonafide': 7355}
  attacks: ['-', 'A07', 'A08', 'A09', 'A10', 'A11', 'A12', 'A13', 'A14', 'A15', 'A16', 'A17', 'A18', 'A19']
ASVspoof2019_LA_dev 24986 flac
ASVspoof2019_LA_eval 71933 flac
ASVspoof2019_LA_train 25380 flac


In [ ]:
import time
t0 = time.time()
!cp "/content/drive/MyDrive/deepfake_finetuning/asvspoof/LA.zip" /content/
print(f'{(time.time()-t0)/60:.1f} min to copy the zip')

t0 = time.time()
!cd /content && unzip -q -o LA.zip
print(f'{(time.time()-t0)/60:.1f} min to unzip')
!find /content/LA -name '*.flac' | wc -l
!df -h /content | tail -1

4.6 min to copy the zip
1.2 min to unzip
122299
overlay         236G   82G  154G  35% /


Feature extraction and hyperparameter selection on held-out training speakers. The five-subsample evaluation printed at the end of this cell is superseded by the single model in the next cell, which is the one reported.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import numpy as np, pandas as pd, librosa, time, json
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, accuracy_score

LA=Path('/content/LA'); OUT=Path('/content/drive/MyDrive/deepfake_finetuning/rebuild_2026')
RES,FEA=OUT/'results',OUT/'features'; SR=16000; WIN=4*SR

def protocol(part,pat):
    f=next(LA.rglob(pat))
    d=pd.DataFrame([l.split() for l in open(f)],columns=['speaker','utt_id','_','attack','label'])
    d['y']=(d.label=='spoof').astype(int)
    sd=next(x for x in LA.rglob(f'ASVspoof2019_LA_{part}') if x.is_dir())
    d['path']=d.utt_id.apply(lambda u: str(sd/'flac'/f'{u}.flac')); return d
tr=protocol('train','*train.trn.txt'); ev=protocol('eval','*eval.trl.txt')

def feat48(p):
    """40 MFCC + 7 spectral contrast + 1 ZCR, 4 s window at 16 kHz (Section 3.3.2)."""
    try:
        y,_=librosa.load(p,sr=SR)
        y=np.pad(y,(0,max(0,WIN-len(y))))[:WIN]
        m=librosa.feature.mfcc(y=y,sr=SR,n_mfcc=40).mean(1)
        sc=librosa.feature.spectral_contrast(y=y,sr=SR,n_bands=6).mean(1)
        z=np.array([librosa.feature.zero_crossing_rate(y).mean()])
        return np.concatenate([m,sc,z]).astype('float32')
    except Exception:
        return np.zeros(48,dtype='float32')

def feats(df,tag):
    f=FEA/f'asvspoof_{tag}_v2.npz'
    if f.exists(): print('cached',f.name); return np.load(f)['X']
    t0=time.time(); out=[]
    with ProcessPoolExecutor(max_workers=2) as ex:
        for i,v in enumerate(ex.map(feat48,df.path.tolist(),chunksize=64),1):
            out.append(v)
            if i%5000==0: print(f'  {tag} {i}/{len(df)} {(time.time()-t0)/60:.0f}m',flush=True)
    X=np.vstack(out); np.savez_compressed(f,X=X,y=df.y.to_numpy(),spk=df.speaker.to_numpy())
    print(f'{tag} {X.shape} in {(time.time()-t0)/60:.0f} min'); return X

Xtr=feats(tr,'train'); Xev=feats(ev,'eval')
print('zero vectors:',int((Xtr==0).all(1).sum()),int((Xev==0).all(1).sum()))

def eer(y,s):
    fpr,tpr,thr=roc_curve(y,s); fnr=1-tpr
    i=int(np.nanargmin(np.abs(fnr-fpr))); return float((fpr[i]+fnr[i])/2), float(thr[i])

rng=np.random.default_rng(42); spk=np.sort(tr.speaker.unique())
sel=set(rng.choice(spk,size=6,replace=False).tolist())
vm=tr.speaker.isin(sel).to_numpy(); fm=~vm
sc0=StandardScaler().fit(Xtr[fm]); Xf,Xv=sc0.transform(Xtr[fm]),sc0.transform(Xtr[vm])
sub=rng.choice(int(fm.sum()),size=min(8000,int(fm.sum())),replace=False)
best,bauc=None,-1
for C in (0.1,1,10,100):
    for gm in ('scale',1e-3,1e-2):
        a=roc_auc_score(tr.y[vm],SVC(kernel='rbf',C=C,gamma=gm,class_weight='balanced')
                        .fit(Xf[sub],tr.y[fm].to_numpy()[sub]).decision_function(Xv))
        print(f'  C={C} gamma={gm}: {a:.4f}',flush=True)
        if a>bauc: best,bauc=dict(C=C,gamma=gm,class_weight='balanced'),a
print('selected',best)

scaler=StandardScaler().fit(Xtr); Xt,Xe=scaler.transform(Xtr),scaler.transform(Xev)
yt,ye,spke=tr.y.to_numpy(),ev.y.to_numpy(),ev.speaker.to_numpy()
runs=[]
for seed in range(5):
    r=np.random.default_rng(seed); ix=r.choice(len(yt),size=12000,replace=False)
    s=SVC(kernel='rbf',**best,random_state=seed).fit(Xt[ix],yt[ix]).decision_function(Xe)
    e,_=eer(ye,s); runs.append(dict(auc=roc_auc_score(ye,s),eer=e,s=s))
    print(f'  seed {seed}: AUC {runs[-1]["auc"]:.4f} EER {e*100:.2f}%',flush=True)
S=np.mean([r['s'] for r in runs],0); e_m,thr=eer(ye,S)
p=(S>thr).astype(int); tn,fp,fn,tp=confusion_matrix(ye,p).ravel()

def boot(fn,n=1000):
    u=np.unique(spke); ix={g:np.where(spke==g)[0] for g in u}; rb=np.random.default_rng(42); o=[]
    for _ in range(n):
        rows=np.concatenate([ix[g] for g in rb.choice(u,size=len(u),replace=True)])
        if len(np.unique(ye[rows]))>1: o.append(fn(ye[rows],S[rows]))
    return float(np.percentile(o,2.5)),float(np.percentile(o,97.5))
alo,ahi=boot(roc_auc_score); elo,ehi=boot(lambda y,s: eer(y,s)[0])

bona=(ye==0); per=[]
for atk in sorted(set(ev.attack[ye==1])):
    m=bona|((ye==1)&(ev.attack.to_numpy()==atk))
    per.append(dict(attack=atk,eer_pct=100*eer(ye[m],S[m])[0]))
per=pd.DataFrame(per).sort_values('eer_pct')

r=dict(features='40 MFCC + 7 spectral contrast + 1 ZCR (48-dim), 4 s window',
       auc=float(roc_auc_score(ye,S)),auc_ci95=[alo,ahi],eer=e_m,eer_ci95=[elo,ehi],
       eer_sd_over_seeds=float(np.std([x['eer'] for x in runs])),
       auc_sd_over_seeds=float(np.std([x['auc'] for x in runs])),
       accuracy_at_eer=float(accuracy_score(ye,p)),
       balanced_accuracy=float((tp/(tp+fn)+tn/(tn+fp))/2),
       spoof_detection_rate=float(tp/(tp+fn)),false_positive_rate=float(fp/(fp+tn)),
       confusion_matrix=[[int(tn),int(fp)],[int(fn),int(tp)]],best_params=str(best),
       n_train=len(yt),n_eval=len(ye))
print('\n'+'\n'.join(f'  {k:22s} {v}' for k,v in r.items()))
print(f"\nEER {e_m*100:.2f}% [{elo*100:.2f}, {ehi*100:.2f}] | AUC {r['auc']:.4f} [{alo:.4f}, {ahi:.4f}]")
print(per.round(2).to_string(index=False))
per.to_csv(RES/'asvspoof_per_attack.csv',index=False)
np.savez_compressed(RES/'asvspoof_eval_scores.npz',y=ye,scores=S,speakers=spke,
                    attacks=ev.attack.to_numpy())
json.dump(r,open(RES/'asvspoof_eval.json','w'),indent=2,default=float)
print('saved')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  train 5000/25380 3m
  train 10000/25380 6m
  train 15000/25380 9m
  train 20000/25380 12m
  train 25000/25380 14m
train (25380, 48) in 14 min
  eval 5000/71237 3m
  eval 10000/71237 6m
  eval 15000/71237 8m
  eval 20000/71237 11m
  eval 25000/71237 14m
  eval 30000/71237 17m
  eval 35000/71237 19m
  eval 40000/71237 22m
  eval 45000/71237 25m
  eval 50000/71237 27m
  eval 55000/71237 30m
  eval 60000/71237 33m
  eval 65000/71237 35m
  eval 70000/71237 38m
eval (71237, 48) in 39 min
zero vectors: 0 0
  C=0.1 gamma=scale: 0.9609
  C=0.1 gamma=0.001: 0.9021
  C=0.1 gamma=0.01: 0.9527
  C=1 gamma=scale: 0.9728
  C=1 gamma=0.001: 0.9519
  C=1 gamma=0.01: 0.9747
  C=10 gamma=scale: 0.9713
  C=10 gamma=0.001: 0.9746
  C=10 gamma=0.01: 0.9666
  C=100 gamma=scale: 0.9715
  C=100 gamma=0.001: 0.9684
  C=100 gamma=0.01: 0.9606
selected {'C': 1, 'gamma': 0.01, 'class_w

Single SVM trained on the full training partition. These are the reported results.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import zipfile, numpy as np, pandas as pd, json, joblib
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score, roc_curve

OUT=Path('/content/drive/MyDrive/deepfake_finetuning/rebuild_2026')
RES,FEA,MOD=OUT/'results',OUT/'features',OUT/'models'
Z='/content/drive/MyDrive/deepfake_finetuning/asvspoof/LA.zip'
z=zipfile.ZipFile(Z)
for n in [
          'LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.eval.trl.txt']:
    z.extract(n,'/content'); print('extracted',n)

tr=np.load(FEA/'asvspoof_train_v2.npz',allow_pickle=True)
ev=np.load(FEA/'asvspoof_eval_v2.npz',allow_pickle=True)
Xtr,ytr=tr['X'],tr['y']; Xev,yev,spke=ev['X'],ev['y'],ev['spk']
sc=StandardScaler().fit(Xtr); Xt,Xe=sc.transform(Xtr),sc.transform(Xev)

BEST=dict(C=1,gamma=0.01,class_weight='balanced')
clf=SVC(kernel='rbf',**BEST).fit(Xt,ytr)          # single model, full training partition
s=clf.decision_function(Xe)

def eer(y,x):
    fpr,tpr,thr=roc_curve(y,x); fnr=1-tpr
    i=int(np.nanargmin(np.abs((1-tpr)-fpr))); return float((fpr[i]+fnr[i])/2),float(thr[i])
e,thr=eer(yev,s)
print(f'single model: EER {e*100:.2f}% | AUC {roc_auc_score(yev,s):.4f} | SVs {clf.n_support_.tolist()}')

atk=np.array([l.split()[3] for l in
              open('/content/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.eval.trl.txt')])
per=[]
for A in sorted(set(atk[yev==1])):
    m=(yev==0)|((yev==1)&(atk==A))
    per.append(dict(attack=A,eer_pct=100*eer(yev[m],s[m])[0]))
per=pd.DataFrame(per).sort_values('eer_pct'); print(per.round(2).to_string(index=False))

def boot(fn,n=1000):
    u=np.unique(spke); ix={g:np.where(spke==g)[0] for g in u}; rb=np.random.default_rng(42); o=[]
    for _ in range(n):
        rows=np.concatenate([ix[g] for g in rb.choice(u,size=len(u),replace=True)])
        if len(np.unique(yev[rows]))>1: o.append(fn(yev[rows],s[rows]))
    return float(np.percentile(o,2.5)),float(np.percentile(o,97.5))
alo,ahi=boot(roc_auc_score); elo,ehi=boot(lambda y,x: eer(y,x)[0])
print(f'\nEER {e*100:.2f}% [{elo*100:.2f}, {ehi*100:.2f}] | AUC {roc_auc_score(yev,s):.4f} [{alo:.4f}, {ahi:.4f}]')

per.to_csv(RES/'asvspoof_per_attack.csv',index=False)
np.savez_compressed(RES/'asvspoof_eval_scores.npz',y=yev,scores=s,speakers=spke,attacks=atk)
joblib.dump({'scaler':sc,'svm':clf},MOD/'svm_asvspoof_single.pkl')
json.dump(dict(model='single SVM trained on the full LA training partition',best_params=str(BEST),
               eer=e,eer_ci95=[elo,ehi],auc=float(roc_auc_score(yev,s)),auc_ci95=[alo,ahi],
               n_support=clf.n_support_.tolist()),
          open(RES/'asvspoof_eval.json','w'),indent=2,default=float)
print('saved')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
extracted LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.eval.trl.txt
single model: EER 16.55% | AUC 0.9186 | SVs [297, 2097]
attack  eer_pct
   A07     0.36
   A08     1.15
   A16     2.15
   A09     2.62
   A19     4.08
   A14     9.51
   A11    12.94
   A10    14.18
   A13    17.11
   A15    23.91
   A12    25.11
   A18    26.20
   A17    33.43

EER 16.55% [14.70, 18.40] | AUC 0.9186 [0.9024, 0.9340]
saved
